In [11]:
import pandas as pd
import numpy as np

# Cargar el DataFrame
df = pd.read_csv('data_clean/DALYs_clean_unificado_NEW.csv', encoding='latin-1')

# =============================================================================
# DICCIONARIO DE DATOS COMPLETO (CON NUEVAS VARIABLES DERIVADAS)
# =============================================================================

# Crear el diccionario de datos como lista de diccionarios
diccionario_data = []

# -----------------------------------------------------------------------------
# 1. VARIABLES BASE (DALYs originales)
# -----------------------------------------------------------------------------

variables_info = [
    # --- IDENTIFICACIÓN ---
    {
        "Variable": "ano",
        "Tipo": str(df["ano"].dtype),
        "Categoría": "Identificación",
        "Descripción": "Año de la estimación de DALYs",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Se extrajo del nombre del archivo original durante el proceso de concatenación. Valores únicos: 2000, 2010, 2015, 2019, 2020, 2021.",
        "Valores/Rango": "2000, 2010, 2015, 2019, 2020, 2021",
        "Uso recomendado": "Variable temporal para análisis de series de tiempo o filtros."
    },
    {
        "Variable": "pais",
        "Tipo": str(df["pais"].dtype),
        "Categoría": "Identificación",
        "Descripción": "Nombre del país en formato largo",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Se transformó de formato ancho (columnas por país) a largo usando pd.melt().",
        "Valores/Rango": "183 países (Afghanistan a Zimbabwe)",
        "Uso recomendado": "Variable de agrupación principal."
    },
    {
        "Variable": "codigo_pais",
        "Tipo": str(df["codigo_pais"].dtype),
        "Categoría": "Identificación",
        "Descripción": "Código ISO alfa-3 del país (estándar internacional de 3 letras)",
        "Fuente": "Global Health Estimates 2021 - OMS / ISO 3166-1 alpha-3",
        "Proceso de Limpieza": "Se agregó mediante mapeo del nombre del país usando diccionario manual 'codigos_paises'.",
        "Valores/Rango": "AFG, ALB, DZA, ... ZWE",
        "Uso recomendado": "Identificador único para merges con otros datasets."
    },
    
    # --- VARIABLES DE ENFERMEDAD ---
    {
        "Variable": "codigo_ghe",
        "Tipo": str(df["codigo_ghe"].dtype),
        "Categoría": "Enfermedad",
        "Descripción": "Código numérico de la enfermedad según la clasificación de la OMS. Filtrado para conservar solo códigos de Enfermedades Tropicales Desatendidas (NTDs).",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Originalmente en float, se convirtió a int. Se filtró usando lista manual de 19 códigos NTDs.",
        "Valores/Rango": "210, 230, 240, 250, 260, 270, 280, 285, 295, 300, 310, 315, 320, 330, 340, 350, 360, 362, 370",
        "Uso recomendado": "Identificador de enfermedad para agrupaciones."
    },
    {
        "Variable": "categoria_principal",
        "Tipo": str(df["categoria_principal"].dtype),
        "Categoría": "Enfermedad",
        "Descripción": "Categoría principal de la enfermedad (nivel superior en jerarquía OMS). En el conjunto filtrado, todas las NTDs pertenecen a la misma categoría principal.",
        "Fuente": "Global Health Estimates 2021 - OMS (catálogo manual)",
        "Proceso de Limpieza": "Valores vacíos completados con forward fill (ffill()) durante la creación del catálogo. Variable constante en el dataset filtrado.",
        "Valores/Rango": "I.Communicable, maternal, perinatal and nutritional conditions",
        "Uso recomendado": "Contexto jerárquico; puede eliminarse para análisis estadístico por ser constante."
    },
    {
        "Variable": "categoria_nivel1",
        "Tipo": str(df["categoria_nivel1"].dtype),
        "Categoría": "Enfermedad",
        "Descripción": "Primer nivel de subcategoría de la enfermedad. En el conjunto filtrado, todas las NTDs pertenecen a la misma subcategoría.",
        "Fuente": "Global Health Estimates 2021 - OMS (catálogo manual)",
        "Proceso de Limpieza": "Valores vacíos completados con forward fill (ffill()) durante la creación del catálogo. Variable constante en el dataset filtrado.",
        "Valores/Rango": "A.Infectious and parasitic diseases",
        "Uso recomendado": "Contexto jerárquico; puede eliminarse para análisis estadístico por ser constante."
    },
    {
        "Variable": "categoria_nivel2",
        "Tipo": str(df["categoria_nivel2"].dtype),
        "Categoría": "Enfermedad",
        "Descripción": "Segundo nivel de subcategoría de la enfermedad. Agrupa las NTDs en tres grandes grupos según la clasificación OMS.",
        "Fuente": "Global Health Estimates 2021 - OMS (catálogo manual)",
        "Proceso de Limpieza": "Valores vacíos completados con forward fill (ffill()) durante la creación del catálogo.",
        "Valores/Rango": "9.Parasitic and vector diseases, 10.Intestinal nematode infections, 12.Other infectious diseases",
        "Uso recomendado": "Agrupación de enfermedades para análisis de alto nivel."
    },
    {
        "Variable": "causa",
        "Tipo": str(df["causa"].dtype),
        "Categoría": "Enfermedad",
        "Descripción": "Nombre específico de la enfermedad o causa. Los valores NaN representan categorías agregadas (no causas específicas), lo cual es esperado por diseño del dataset original.",
        "Fuente": "Global Health Estimates 2021 - OMS (catálogo manual)",
        "Proceso de Limpieza": "Se mantuvieron los NaN para diferenciar categorías agregadas de causas específicas.",
        "Valores/Rango": "b.Trypanosomiasis, c.Chagas disease, e.Leishmaniasis, a.Ascariasis, ... (19 causas específicas + NaN)",
        "Uso recomendado": "Variable principal de enfermedad para análisis detallados."
    },
    
    # --- VARIABLES DEMOGRÁFICAS ---
    {
        "Variable": "sexo",
        "Tipo": str(df["sexo"].dtype),
        "Categoría": "Demografía",
        "Descripción": "Sexo de la población afectada",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Se procesaron los datos de hombres y mujeres por separado (hojas diferentes del Excel) y luego se combinaron usando merge.",
        "Valores/Rango": "Males, Females",
        "Uso recomendado": "Variable de estratificación para análisis por género."
    },
    {
        "Variable": "edad",
        "Tipo": str(df["edad"].dtype),
        "Categoría": "Demografía",
        "Descripción": "Rango de edad de la población afectada. Los datos originales vienen en hojas separadas por grupo etario.",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Se extrajo del nombre de las hojas del Excel original durante el proceso de iteración.",
        "Valores/Rango": "0-4, 5-14, 15-29, 30-49, 50-59, 60-69, 70+",
        "Uso recomendado": "Variable de estratificación para análisis por grupos de edad."
    },
    {
        "Variable": "poblacion_miles",
        "Tipo": str(df["poblacion_miles"].dtype),
        "Categoría": "Demografía",
        "Descripción": "Población total, en miles de personas, para el grupo específico de sexo y edad.",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Extraído de filas específicas (227 para hombres, 445 para mujeres) de cada hoja Excel, luego pivotado a formato largo con pd.melt() y unido con merge usando año, código_pais, sexo y edad.",
        "Valores/Rango": "0.001 a 1,000,000+",
        "Uso recomendado": "Denominador para calcular tasas poblacionales."
    },
    
    # --- VARIABLE DEPENDIENTE PRINCIPAL ---
    {
        "Variable": "DALYs",
        "Tipo": str(df["dalys"].dtype),
        "Categoría": "Carga de enfermedad",
        "Descripción": "Años de Vida Ajustados por Discapacidad (Disability-Adjusted Life Years). Métrica que combina años de vida perdidos por muerte prematura (YLL) y años vividos con discapacidad (YLD).",
        "Fuente": "Global Health Estimates 2021 - OMS",
        "Proceso de Limpieza": "Se transformó de formato ancho (columnas por país) a largo mediante pd.melt(). Sin valores nulos.",
        "Valores/Rango": "~10⁻⁹ a ~10²",
        "Uso recomendado": "Variable dependiente principal."
    },

    # -----------------------------------------------------------------------------
    # 2. VARIABLES SOCIOECONÓMICAS
    # -----------------------------------------------------------------------------
    {
        "Variable": "intervenciones_NTDs",
        "Tipo": str(df["intervenciones_ntds"].dtype),
        "Categoría": "Socioeconómica - Salud",
        "Descripción": "Número de personas que requirieron tratamiento para cualquier enfermedad clasificada como NTD según el roadmap de la OMS, reportado a la OMS.",
        "Fuente": "World Health Organization - Global Health Observatory (2025) – processed by Our World in Data",
        "Proceso de Limpieza": "Se realizó merge (left join) con dataset de OWID usando país y año como llaves.",
        "Valores/Rango": "0 a ~10⁸",
        "Uso recomendado": "Proxy de recursos destinados a combatir NTDs."
    },
    {
        "Variable": "GDP_per_capita",
        "Tipo": str(df["pib_per_capita"].dtype),
        "Categoría": "Socioeconómica - Economía",
        "Descripción": "Producto Interno Bruto per cápita ajustado por inflación y paridad de poder adquisitivo (PPP). Medido en dólares internacionales constantes.",
        "Fuente": "Our World in Data / Banco Mundial",
        "Proceso de Limpieza": "Se realizó merge (left join) con dataset de OWID usando país y año. Se conservó a pesar de algunos valores NaN.",
        "Valores/Rango": "~500 a ~100,000",
        "Uso recomendado": "Indicador de riqueza/desarrollo económico."
    },
    {
        "Variable": "coeficiente_gini",
        "Tipo": str(df["coeficiente_gini"].dtype) if "coeficiente_gini" in df.columns else "No disponible",
        "Categoría": "Socioeconómica - Desigualdad",
        "Descripción": "Coeficiente de Gini - medida de desigualdad en la distribución del ingreso (0 = perfecta igualdad, 1 = perfecta desigualdad). Medido sobre ingresos después de impuestos y beneficios.",
        "Fuente": "World Inequality Database (WID.world) (2025) – processed by Our World in Data",
        "Proceso de Limpieza": "Se realizó merge con dataset de WID usando país y año. **Variable eliminada** en versión final por alta presencia de NaN (>60%).",
        "Valores/Rango": "0.2 - 0.7 típicamente",
        "Uso recomendado": "No aplica (eliminada del dataset final)."
    },

    # -----------------------------------------------------------------------------
    # 3. VARIABLES DE DESARROLLO HUMANO
    # -----------------------------------------------------------------------------
    {
        "Variable": "hdi",
        "Tipo": str(df["idh"].dtype),
        "Categoría": "Desarrollo Humano",
        "Descripción": "Índice de Desarrollo Humano (IDH) subnacional. Medida resumen del logro promedio en salud (esperanza de vida), educación (años de escolaridad) e ingresos (INB per cápita). Escala 0-1.",
        "Fuente": "Global Data Lab (GDL), Subnational Human Development Index (SHDI). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Se cargaron archivos separados por sexo (hdi_masc, hdi_fem), se unificaron y se añadió mediante merge con país, año y sexo.",
        "Valores/Rango": "0.3 - 0.95",
        "Uso recomendado": "Indicador compuesto de desarrollo."
    },
    {
        "Variable": "indice_educacion",
        "Tipo": str(df["indice_educacion"].dtype),
        "Categoría": "Desarrollo Humano",
        "Descripción": "Índice de educación, componente del IDH. Combina años promedio de escolaridad y años esperados de escolaridad.",
        "Fuente": "Global Data Lab (GDL). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Creado a partir de 'edindexm' (Males) y 'edindexf' (Females) usando np.where según la columna 'sexo'.",
        "Valores/Rango": "0.2 - 0.9",
        "Uso recomendado": "Análisis de componente específico del desarrollo."
    },
    {
        "Variable": "indice_salud",
        "Tipo": str(df["indice_salud"].dtype),
        "Categoría": "Desarrollo Humano",
        "Descripción": "Índice de salud, componente del IDH. Basado en esperanza de vida al nacer.",
        "Fuente": "Global Data Lab (GDL). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Creado a partir de 'healthindexm' y 'healthindexf' usando np.where según la columna 'sexo'.",
        "Valores/Rango": "0.4 - 0.95",
        "Uso recomendado": "Análisis de componente específico del desarrollo."
    },
    {
        "Variable": "indice_ingresos",
        "Tipo": str(df["indice_ingresos"].dtype),
        "Categoría": "Desarrollo Humano",
        "Descripción": "Índice de ingresos, componente del IDH. Basado en INB per cápita (PPA en USD).",
        "Fuente": "Global Data Lab (GDL). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Creado a partir de 'incindexm' y 'incindexf' usando np.where según la columna 'sexo'.",
        "Valores/Rango": "0.3 - 0.9",
        "Uso recomendado": "Análisis de componente específico del desarrollo."
    },
    {
        "Variable": "esperanza_vida",
        "Tipo": str(df["esperanza_vida"].dtype),
        "Categoría": "Desarrollo Humano",
        "Descripción": "Esperanza de vida al nacer, en años. Dato utilizado para construir el índice de salud del IDH.",
        "Fuente": "Global Data Lab (GDL). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Creado a partir de 'lifexpm' y 'lifexpf' usando np.where según la columna 'sexo'.",
        "Valores/Rango": "45 - 85 años",
        "Uso recomendado": "Indicador directo de salud poblacional."
    },

    # -----------------------------------------------------------------------------
    # 4. VARIABLES AMBIENTALES
    # -----------------------------------------------------------------------------
    {
        "Variable": "co2_anual",
        "Tipo": str(df["co2_anual"].dtype),
        "Categoría": "Ambiental",
        "Descripción": "Emisiones anuales totales de CO2, en toneladas.",
        "Fuente": "Global Data Lab (GDL). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Se realizó merge (left join) tras limpieza manual del archivo 'GDL-Total-Yearly-CO2-Emissions-(ton)-data.csv'.",
        "Valores/Rango": "0 a ~10¹⁰",
        "Uso recomendado": "Indicador de actividad industrial/contaminación."
    },
    {
        "Variable": "precipitacion_anual",
        "Tipo": str(df["precipitacion_anual"].dtype) if "precipitacion_anual" in df.columns else "float64",
        "Categoría": "Ambiental",
        "Descripción": "Precipitación total anual (lluvia y nieve) en metros (m). Calculada como la suma de promedios diarios, reportada como profundidad de agua en superficie.",
        "Fuente": "Global Data Lab (GDL), procesa datos de Copernicus Climate Change Service (2025). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Se realizó merge (left join) tras limpieza manual del archivo 'GDL-Total-Yearly-Precipitation-(m)-data.csv'.",
        "Valores/Rango": "0 - 5 metros",
        "Uso recomendado": "Variable climática, relevante para enfermedades transmitidas por vectores."
    },
    {
        "Variable": "temperatura_superficial",
        "Tipo": str(df["temperatura_superficial"].dtype) if "temperatura_superficial" in df.columns else "float64",
        "Categoría": "Ambiental",
        "Descripción": "Temperatura superficial promedio anual, en grados Celsius.",
        "Fuente": "Global Data Lab (GDL), procesa datos de Copernicus Climate Change Service (2025). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Se realizó merge (left join) tras limpieza manual del archivo 'GDL-Yearly-Average-Surface-Temperature-(ºC)-data.csv'.",
        "Valores/Rango": "-20 a 35 °C",
        "Uso recomendado": "Variable climática clave para enfermedades sensibles a temperatura."
    },
    {
        "Variable": "humedad_relativa",
        "Tipo": str(df["humedad_relativa"].dtype) if "humedad_relativa" in df.columns else "float64",
        "Categoría": "Ambiental",
        "Descripción": "Humedad relativa promedio anual, en porcentaje.",
        "Fuente": "Global Data Lab (GDL), procesa datos de Copernicus Climate Change Service (2025). https://globaldatalab.org. Uso no comercial.",
        "Proceso de Limpieza": "Se realizó merge (left join) tras limpieza manual del archivo 'GDL-Yearly-Average-Relative-Humidity-(_)-data.csv'.",
        "Valores/Rango": "20% - 90%",
        "Uso recomendado": "Variable climática, relevante para supervivencia de vectores."
    },

    # -----------------------------------------------------------------------------
    # 5. VARIABLES DERIVADAS
    # -----------------------------------------------------------------------------
    {
        "Variable": "poblacion_abs",
        "Tipo": str(df["poblacion_abs"].dtype) if "poblacion_abs" in df.columns else "float64",
        "Categoría": "Derivada - Demografía",
        "Descripción": "Población absoluta (no en miles) calculada para facilitar cálculos de tasas. Equivale a población real.",
        "Fuente": "Derivada de Global Health Estimates 2021",
        "Proceso de Limpieza": "Creada mediante la operación: poblacion_abs = poblacion_miles × 1000",
        "Valores/Rango": "1 a 10⁹",
        "Uso recomendado": "Variable auxiliar para cálculos; no usar directamente en modelos."
    },
    {
        "Variable": "tasa_dalys_100k",
        "Tipo": str(df["tasa_dalys_100k"].dtype) if "tasa_dalys_100k" in df.columns else "float64",
        "Categoría": "Derivada - Carga de enfermedad",
        "Descripción": "Tasa de DALYs por cada 100,000 habitantes. Variable fundamental que permite comparar la carga de la enfermedad entre países o regiones con diferentes tamaños de población, eliminando el efecto del tamaño poblacional.",
        "Fuente": "Derivada de Global Health Estimates 2021",
        "Proceso de Limpieza": "Calculada como: (DALYs / poblacion_abs) × 100,000. Esta estandarización es esencial para análisis comparativos.",
        "Valores/Rango": "0 a ~10⁵",
        "Uso recomendado": "Variable dependiente principal para comparaciones geográficas. Más interpretable que DALYs absolutos."
    },
    {
        "Variable": "dalys_por_intervencion",
        "Tipo": str(df["dalys_por_intervencion"].dtype) if "dalys_por_intervencion" in df.columns else "float64",
        "Categoría": "Derivada - Eficiencia",
        "Descripción": "Relación entre los DALYs y el número de intervenciones contra NTDs. Métrica de 'eficiencia inversa': un valor alto puede indicar que, a pesar de muchas intervenciones, la carga de la enfermedad persiste.",
        "Fuente": "Derivada de Global Health Estimates 2021 y WHO GHO",
        "Proceso de Limpieza": "Calculada como DALYs / (intervenciones_NTDs + 1). Se suma 1 al denominador para evitar división por cero en casos sin intervenciones.",
        "Valores/Rango": "0 a ~10⁻⁵",
        "Uso recomendado": "Análisis de efectividad de intervenciones sanitarias."
    },
    {
        "Variable": "desarrollo_vs_carga",
        "Tipo": str(df["desarrollo_vs_carga"].dtype) if "desarrollo_vs_carga" in df.columns else "float64",
        "Categoría": "Derivada - Interacción",
        "Descripción": "Variable de interacción entre el Índice de Desarrollo Humano (hdi) y la tasa de DALYs. Captura el efecto conjunto del nivel de desarrollo y la carga de la enfermedad.",
        "Fuente": "Derivada de Global Data Lab (GDL) y Global Health Estimates 2021",
        "Proceso de Limpieza": "Calculada como el producto de 'hdi' y 'tasa_dalys_100k'. Útil para modelos predictivos que buscan incluir efectos de interacción.",
        "Valores/Rango": "0 a ~10⁵",
        "Uso recomendado": "Variable de interacción para modelos de regresión múltiple."
    }
]

# Crear DataFrame del diccionario
df_diccionario = pd.DataFrame(variables_info)

csv_path = 'data_clean/Diccionario_Datos_DALYs.csv'
df_diccionario.to_csv(csv_path, index=False)